In [3]:
# ============================================================
# PRÉDICTION DU TAUX DE RÉUSSITE - VERSION MOYENNE
# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

print("="*60)
print("PRÉDICTION DU TAUX DE RÉUSSITE - VERSION MOYENNE")
print("="*60)

# 1. CHARGEMENT
print("\n1. CHARGEMENT...")
df_diplomes = pd.read_csv('../data/fact_diplomes.csv', sep=';', encoding='utf-8')
df_inscrits = pd.read_csv('../data/fact_inscrits.csv', sep=';', encoding='utf-8')

# 2. NETTOYAGE (garder moins de données)
print("\n2. NETTOYAGE...")
df_diplomes = df_diplomes.drop_duplicates().dropna()
df_inscrits = df_inscrits.drop_duplicates().dropna()

# Garder seulement 70% des données
df_diplomes = df_diplomes.sample(frac=0.7, random_state=42)
df_inscrits = df_inscrits.sample(frac=0.7, random_state=42)
print(f"   Diplômes: {len(df_diplomes)} lignes")
print(f"   Inscrits: {len(df_inscrits)} lignes")

# 3. AGRÉGATION
print("\n3. AGRÉGATION...")
df_diplomes_agg = df_diplomes.groupby(['universite_code','etablissement_code']).agg({
    'diplomes_M':'sum','diplomes_F':'sum','diplomes_total':'sum'
}).reset_index()

df_inscrits_2022 = df_inscrits[df_inscrits['annee'] == 2022].copy()
df_inscrits_2022['inscrits_total'] = df_inscrits_2022['inscrits_f'] + df_inscrits_2022['inscrits_m']
df_inscrits_agg = df_inscrits_2022.groupby(['code_universite','code_etablissement']).agg({
    'inscrits_f':'sum','inscrits_m':'sum','inscrits_total':'sum'
}).reset_index()

# 4. FUSION
print("\n4. FUSION...")
df_diplomes_agg = df_diplomes_agg.rename(columns={'universite_code':'univ_code','etablissement_code':'etab_code'})
df_inscrits_agg = df_inscrits_agg.rename(columns={'code_universite':'univ_code','code_etablissement':'etab_code'})
df = pd.merge(df_diplomes_agg, df_inscrits_agg, on=['univ_code','etab_code'], how='inner')
print(f"   Fusion: {len(df)} lignes")

# 5. CIBLE
print("\n5. CRÉATION DE LA CIBLE...")
df['taux_reussite'] = (df['diplomes_total'] / df['inscrits_total']).clip(0, 1)
print(f"   Taux moyen: {df['taux_reussite'].mean():.2%}")

# 6. FEATURES (peu nombreuses)
print("\n6. CRÉATION DES FEATURES...")
df['ratio_feminisation'] = df['inscrits_f'] / df['inscrits_total']
df['taille_cohorte'] = df['inscrits_total']

features = ['ratio_feminisation', 'taille_cohorte']
X = df[features]
y = df['taux_reussite']

# 7. MODÈLE (peu optimisé)
print("\n7. MODÈLE...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42)
model.fit(X_train, y_train)

# 8. RÉSULTATS
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*60)
print("RÉSULTATS")
print("="*60)
print(f"   R² = {r2:.4f}")
print(f"   MAE = {mae*100:.1f}%")
print("="*60)

PRÉDICTION DU TAUX DE RÉUSSITE - VERSION MOYENNE

1. CHARGEMENT...

2. NETTOYAGE...
   Diplômes: 1422 lignes
   Inscrits: 12391 lignes

3. AGRÉGATION...

4. FUSION...
   Fusion: 175 lignes

5. CRÉATION DE LA CIBLE...
   Taux moyen: 28.51%

6. CRÉATION DES FEATURES...

7. MODÈLE...

RÉSULTATS
   R² = -0.0254
   MAE = 13.1%
